# Full Pipeline (Modulaire)

Ce notebook est découpé en etapes pour tester chaque morceau individuellement:
1. Telechargement Kaggle
2. Scraping Reddit
3. Nettoyage + balancing
4. Training Logistic Regression et XGBoost
5. Metriques (accuracy, precision, recall, f1).

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import config
from src.data_cleaning import download_data
from src.data_cleaning.balance_classes import balance_classes, remove_label0_kaggle
from src.data_cleaning.clean_data import add_label0, clean_data_kaggle, clean_data_scrapped
from src.training.evaluate import evaluate
from src.training.train import prepare_data, save_artifacts, train_log_model, train_xgboost_model

In [ ]:
# Changez les valeurs en False si vous ne voulez pas exécuter une étape lors du Run All
RUN_KAGGLE_DOWNLOAD = True
RUN_REDDIT_SCRAPING = True
RUN_CLEAN_BALANCE = True
RUN_TRAIN_LR = True
RUN_TRAIN_XGB = True

In [ ]:
# Parametres utiles pour des tests rapides
MAX_POSTS_OVERRIDE = None  # ex: 100 pour test rapide
RANDOM_STATE = 42

RAW_SCRAPED_PATH = PROJECT_ROOT / 'data' / 'raw' / 'happiness_reddit.csv'
FINAL_DATASET_PATH = PROJECT_ROOT / 'data' / 'cleaned' / 'balanced_30k_dataset.csv'

RAW_SCRAPED_PATH.parent.mkdir(parents=True, exist_ok=True)
FINAL_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Final dataset path:', FINAL_DATASET_PATH)

## Etape 1 - Telechargement Kaggle

In [ ]:
if RUN_KAGGLE_DOWNLOAD:
    download_data.download_data_kaggle()

kaggle_csv_path = config.DATA_DIR / config.DATA_FILENAME
print('Kaggle csv path:', kaggle_csv_path)
print('Exists:', kaggle_csv_path.exists())

## Etape 2 - Chargement + nettoyage donnees Kaggle

In [ ]:
df_kaggle = pd.read_csv(kaggle_csv_path)
df_kaggle = remove_label0_kaggle(df_kaggle)
df_dep_clean = clean_data_kaggle(df_kaggle)

print('Kaggle raw shape:', df_kaggle.shape)
print('Kaggle cleaned shape:', df_dep_clean.shape)
df_dep_clean.head()

## Etape 3 - Scraping Reddit

In [ ]:
original_max_posts = download_data.MAX_POSTS_PER_SUBREDDIT
if MAX_POSTS_OVERRIDE is not None:
    download_data.MAX_POSTS_PER_SUBREDDIT = MAX_POSTS_OVERRIDE

try:
    if RUN_REDDIT_SCRAPING:
        df_scraped = download_data.all_posts_listed()
        download_data.save_posts_to_csv(df_scraped, output_path=RAW_SCRAPED_PATH)
    else:
        df_scraped = pd.read_csv(RAW_SCRAPED_PATH)
finally:
    download_data.MAX_POSTS_PER_SUBREDDIT = original_max_posts

print('Scraped shape:', df_scraped.shape)
df_scraped.head()

## Etape 4 - Nettoyage scraping + balancing + sauvegarde dataset final

In [ ]:
if RUN_CLEAN_BALANCE:
    df_scraped = add_label0(df_scraped)
    df_happy_clean = clean_data_scrapped(df_scraped)
    df_balanced = balance_classes(df_dep_clean, df_happy_clean, random_state=RANDOM_STATE)
    df_balanced.to_csv(FINAL_DATASET_PATH, index=False)
else:
    df_balanced = pd.read_csv(FINAL_DATASET_PATH)

print('Final balanced shape:', df_balanced.shape)
print(df_balanced['label'].value_counts())
df_balanced.head()

## Etape 5 - Split train/val/test

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = prepare_data(df_cleaned=df_balanced)

print('Train size:', len(X_train))
print('Val size:', len(X_val))
print('Test size:', len(X_test))

## Etape 6 - Training + evaluation Logistic Regression

In [ ]:
lr_metrics = None
if RUN_TRAIN_LR:
    lr_vectorizer, lr_model = train_log_model(X_train, y_train)
    save_artifacts(lr_vectorizer, lr_model)
    lr_metrics = evaluate(lr_model, lr_vectorizer, X_test, y_test)
    print('LR metrics:')
    print({
        'accuracy': round(lr_metrics['accuracy'], 4),
        'precision': round(lr_metrics['precision'], 4),
        'recall': round(lr_metrics['recall'], 4),
        'f1_score': round(lr_metrics['f1_score'], 4),
    })

## Etape 7 - Training + evaluation XGBoost

In [ ]:
xgb_metrics = None
if RUN_TRAIN_XGB:
    xgb_vectorizer, xgb_model = train_xgboost_model(X_train, y_train)
    save_artifacts(
        xgb_vectorizer,
        xgb_model,
        vectorizer_path=config.XGBOOST_VECTORIZER_PATH,
        model_path=config.XGBOOST_MODEL_PATH,
    )
    xgb_metrics = evaluate(xgb_model, xgb_vectorizer, X_test, y_test)
    print('XGBoost metrics:')
    print({
        'accuracy': round(xgb_metrics['accuracy'], 4),
        'precision': round(xgb_metrics['precision'], 4),
        'recall': round(xgb_metrics['recall'], 4),
        'f1_score': round(xgb_metrics['f1_score'], 4),
    })

## Resume final

In [ ]:
summary = {}
if lr_metrics is not None:
    summary['logistic_regression'] = {
        'accuracy': lr_metrics['accuracy'],
        'precision': lr_metrics['precision'],
        'recall': lr_metrics['recall'],
        'f1_score': lr_metrics['f1_score'],
    }
if xgb_metrics is not None:
    summary['xgboost'] = {
        'accuracy': xgb_metrics['accuracy'],
        'precision': xgb_metrics['precision'],
        'recall': xgb_metrics['recall'],
        'f1_score': xgb_metrics['f1_score'],
    }

summary